# Step 3.5: the indicators

This notebook is the check of step 3.5. `ff.indicators` holds the common indicators as features: moving averages, RSI, ATR, Bollinger bands, MACD, the stochastic, ADX, and more, each giving the numbers TA-Lib gives. Here you compute the 14-bar RSI on a month of 5-minute bars and compare three of its values with your charting platform's RSI on the same bars. TradingView's and NinjaTrader 8's RSI both use Wilder's smoothing, as TA-Lib's does, and all of them agree once a few hundred bars have passed, so the three bars are on the month's last trade date.

Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR` and keeps the RSI's values in the feature cache under `FEFAT_HOME`.

## RSI(14) on a month of 5-minute bars

The product is MES when your store holds its 1-minute bars, else the product with the most contracts, and the window the 30 days to the last trade date of its data.

Expected: the product, the window, and the last trade date whose data reaches 14:00 Chicago time, with its contract; then three bars of that date, opening at 09:00, 12:00, and 14:00 Chicago time, each with its close and its RSI(14) to four decimals. On your chart, open that contract at 5 minutes, add RSI with a length of 14 on the close, and read it on the same three bars: TradingView labels a bar by its open, NinjaTrader 8 by its close. The values match to the second decimal.

In [ ]:
from datetime import timedelta

import polars as pl

import fefat as ff
from fefat.market.engine_units import price_text

listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
FIRST = LAST - timedelta(days=30)
computed = ff.features.compute({"rsi": ff.indicators.rsi(14)}, ROOT, "5m", FIRST, LAST).collect()

chicago = {
    name: pl.col(name)
    .cast(pl.Datetime("ns", "UTC"))
    .dt.convert_time_zone("America/Chicago")
    .dt.strftime("%H:%M")
    for name in ("ts_open", "ts_close")
}
at_times = computed.with_columns(opens=chicago["ts_open"], closes=chicago["ts_close"]).filter(
    pl.col("opens").is_in(["09:00", "12:00", "14:00"])
)
DAY = at_times.filter(pl.col("opens") == "14:00").get_column("trade_date").max()
shown = at_times.filter(pl.col("trade_date") == DAY)
contract = shown.item(0, "contract")
print(f"{ROOT} 5-minute bars from {FIRST} to {LAST}, contract {contract} on {DAY}")
for bar in shown.iter_rows(named=True):
    close, rsi = price_text(bar["close"]), bar["rsi"]
    print(f"{bar['opens']} to {bar['closes']} Chicago: close {close}, RSI(14) {rsi:.4f}")

## The same RSI from TA-Lib

TA-Lib, the reference library of technical indicators, computes RSI(14) from the window's closes alone, where `compute` also read the sessions before the window. Past the indicator's warm-up of 350 bars the two forget where they started.

Expected: the number of bars compared and the largest difference, below 1e-7, a billionth of RSI's range of 100, and `PASSED`.

In [ ]:
import numpy as np
import talib

closes = ff.market.load(ROOT, "5m", FIRST, LAST).collect().get_column("close")
theirs = talib.RSI(closes.cast(pl.Float64).to_numpy(), 14)
ours = computed.get_column("rsi").cast(pl.Float64).to_numpy()
past = ff.indicators.rsi(14).warmup
gap = np.abs(ours[past:] - theirs[past:]).max()
verdict = "PASSED" if gap <= 1e-7 else "FAILED"
compared = len(ours) - past
print(f"RSI(14) and TA-Lib's on the {compared:,} bars past the warm-up: {gap:.1e} apart: {verdict}")